# Combine USGS Site Data
This notebook takes the CKAN/Upstream-backed CSV files created by the ingest notebook, extracts the site metadata table, standardizes site names, and combines the individual site histories into one master compaction dataset.


## Before You Run

- Run `1_compaction_data_ingest.ipynb` first, or confirm that the expected files already exist in `csv_files/`.
- Check that `csv_files/TABLE1_CompactionSites.csv` and the per-station measurement CSV files are present.
- Keep the CKAN registration checkpoint in mind: the cleaned CSV outputs from this notebook are intended to become reusable catalog resources.

## Expected Outputs

- `compaction_sites.csv` contains cleaned station metadata for map markers.
- `combined_site_data.csv` contains the long-form compaction time series used by the Folium notebook.
- The notebook provides example CKAN dataset and resource metadata for registering those cleaned CSV files.


In [1]:
import os
import pandas as pd

### Load the Converted CSV Files and Site Metadata
List the CSV files created in the previous notebook, separate the main site metadata table from the site-specific measurement files, and create a condensed site-name field that is easier to match later.


In [2]:
# Get the CKAN/Upstream-backed CSV files created by the ingest notebook
directory_path = "csv_files"
files = [f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]

# pull out the table that actually has the information for the sites
sites_file = 'TABLE1_CompactionSites.csv'
files.remove(sites_file)

# Read sites into data frame
sites = pd.read_csv(os.path.join(directory_path, sites_file))

# add column to mathch the site name pulled from the site specific data file (i.e. no spaces)
sites['name_condensed'] = sites.apply(lambda x: x['GENERAL_NM'].replace(" ",""), axis=1)

### Build a Simple Site Name List
Pull the unique condensed site names into a separate DataFrame so you can review the set of valid site identifiers.


In [3]:
sites_names = sites.name_condensed.unique()
sites_df = pd.DataFrame(sites_names, columns = ['name'])

### Save the Site Metadata Table
Write the cleaned site metadata to `compaction_sites.csv` so it can be reused by the mapping notebook.


In [4]:
sites.to_csv('compaction_sites.csv')

### Preview the Site Metadata
Display the first few rows to confirm the site table looks correct and includes the new condensed-name field.


In [5]:
sites.head()

,SITE_NO,STATION_NM,COMPACTION_INTERVAL,ANCHOR_DEPTH,DEC_LONG_VA,DEC_LAT_VA,GENERAL_NM,name_condensed
0,294726095351102,LJ-65-12-726 (Addicks Extensometer),CHICOT AND EVANGELINE,1802,-95.5861,29.7907,Addicks,Addicks
1,292458094534206,KH-64-33-920 (Texas City Extensometer),CHICOT,800,-94.8950,29.4163,Texas City,TexasCity
2,294338095270402,LJ-65-21-226 (Southwest Extensometer),CHICOT AND EVANGELINE,2358,-95.4508,29.7270,Southwest,Southwest
3,293352095011601,LJ-65-32-625 (Seabrook Extensometer),CHICOT,1381,-95.0215,29.5648,Seabrook,Seabrook
4,294237095093204,LJ-65-23-322 (Pasadena Extensometer),CHICOT AND EVANGELINE,2831,-95.1593,29.7102,Pasadena,Pasadena


## Combine the Site Measurement Files
The remaining steps focus on stacking the individual site CSV files into one long-form dataset that contains dates, compaction values, site names, and a simple version label.


### Check File Names Against Known Sites
Inspect the measurement filenames and compare the parsed site name against the metadata-derived list. This is a quick validation step to catch inconsistent file naming.


In [6]:
for file in files:
    file_info = file.split('_')
    if file_info[1] not in sites_names:
        print('error: ', file)
        print(file_info)

error:  TABLE13_BaytownC1_Shallow_2023.csv
['TABLE13', 'BaytownC1', 'Shallow', '2023.csv']
error:  TABLE14_BaytownC2_Deep_2023.csv
['TABLE14', 'BaytownC2', 'Deep', '2023.csv']


### Standardize Special Cases and Concatenate the Data
Handle a few site names that need manual replacement, read each site CSV, attach the site name and data version from the filename, and append everything into one combined DataFrame.


In [7]:
name_replace_dict = {
    'BaytownC2_Deep': 'BaytownDeep',
    'BaytownC1_Shallow': 'BaytownShallow'
}
sites_data = pd.DataFrame(columns = ['DATE','CUMULATIVE_COMPACTION','site','data_version'])
for file in files:
    file_info = file
    for name in name_replace_dict.keys():
        if name in file:
            file_info = file_info.replace(name, name_replace_dict[name])         
    site_info = file_info.split('_')
    site_name = site_info[1]
    data_year = site_info[-1].replace('.csv','')
    df = pd.read_csv(os.path.join('csv_files',file))
    df['site'] = site_name
    df['data_version'] = data_year
    sites_data = pd.concat([sites_data, df], ignore_index=True)

/var/folders/ps/dx2yrk_1117grf32kqlw9qyh0000gq/T/ipykernel_15698/1850334796.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sites_data = pd.concat([sites_data, df], ignore_index=True)


### Save the Combined Time-Series Dataset
Write the final long-form compaction table to `combined_site_data.csv` for use in the plotting and Folium notebooks.


In [8]:
sites_data.to_csv('combined_site_data.csv')

## Register the Cleaned Data in CKAN

At this stage the two core outputs are ready for publication:

- `compaction_sites.csv` contains the station metadata used for map markers
- `combined_site_data.csv` contains the cleaned long-form time-series measurements

This is the best point in the workflow to register data with the `ckan-jupyter` extension. The files are stable enough to describe and publish, but you have not yet created downstream presentation artifacts such as popup HTML files or the final Folium map.


### Why Register Here?

Registering the data now helps other people, tools, and future you understand what this dataset is and how it should be used. A registered dataset is easier to find, easier to trust, and much easier to reuse in later notebooks or portal workflows.

Participants should register the cleaned CSV files because registration:

- makes the dataset discoverable instead of leaving it in one local folder
- preserves the context needed to interpret the measurements later
- provides enough description for someone else to reuse the data without hearing the live explanation
- creates a clean handoff from data preparation into mapping, portal, and workflow steps

The `ckan-jupyter` extension currently asks for dataset metadata fields:

- `Title`
- `Description`
- `Owner Organization`
- `License ID`
- `Source URL`
- `Tags`
- `Spatial Coverage`

The dataset `Name` is generated automatically from the title, so participants do not need to type it manually.

Example dataset metadata participants can copy and adapt:

| Field | Example Value |
| --- | --- |
| Title | `Houston-area extensometer compaction measurements` |
| Description | `Cleaned cumulative compaction measurements combined from multiple USGS extensometer site files for the DSO Day 3 Folium mapping exercise.` |
| Owner Organization | `Select the workshop organization assigned for class` |
| License ID | `notspecified` |
| Source URL | `https://www.usgs.gov/` |
| Tags | `compaction, extensometer, groundwater, texas, folium, usgs` |
| Spatial Coverage | `{"type":"Polygon","coordinates":[[[-95.9,29.1],[-94.8,29.1],[-94.8,30.3],[-95.9,30.3],[-95.9,29.1]]]}` |

The same workflow also asks for data resource metadata when the file is uploaded to the dataset:

- `Name`
- `Description`
- `Format`
- `Jupyter File`

Example resource metadata for `combined_site_data.csv`:

| Field | Example Value |
| --- | --- |
| Name | `combined_site_data.csv` |
| Description | `Cleaned long-form cumulative compaction time-series data combined from all site CSV files used in the 2nd Folium exercise.` |
| Format | `CSV` |
| Jupyter File | `combined_site_data.csv` |

Example resource metadata for `compaction_sites.csv`:

| Field | Example Value |
| --- | --- |
| Name | `compaction_sites.csv` |
| Description | `Station metadata table containing site names and coordinates used to place markers on the Folium map.` |
| Format | `CSV` |
| Jupyter File | `compaction_sites.csv` |

Notes for participants:

- The dataset `Name` is auto-generated from `Title` in the Jupyter form.
- `Tags` should be comma-separated in the Jupyter form.
- `Spatial Coverage` should be valid GeoJSON, not a plain-text county list.
- Resource metadata describes the uploaded file itself, not the dataset as a whole.

A practical handoff is:
1. inspect the cleaned outputs in this notebook
2. create the dataset record with the `ckan-jupyter` extension
3. upload one or both CSV files with resource metadata
4. continue into the Folium notebook using those cleaned products


### Result
You now have two reusable outputs: `compaction_sites.csv` for marker metadata and `combined_site_data.csv` for the compaction time series plotted in map popups.
